Cleaning data from CDC Social Vulnerability Index to merge with geocoded restaurant addresses



In [11]:
#imports
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [ ]:
#loading CDC Social Vulnerability Index data that is downloaded at the ZCTA level
df = pd.read_csv("/Users/pdeguz01/Documents/git/Data/IDS705_Final/SVI_2022_US_ZCTA.csv")


In [9]:
#loading geocoded address data
rest_df = pd.read_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_geocoded.csv")

/var/folders/81/w_61xz297rv4ggdktb58tlxm0000gn/T/ipykernel_56931/1805861460.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  rest_df = pd.read_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_geocoded.csv")


In [15]:
#calculate primary ZCTA for each geocoded address
#Creating geometry column
geometry = gpd.points_from_xy(rest_df.Longitude, rest_df.Latitude)
gdf = gpd.GeoDataFrame(rest_df, geometry=geometry, crs="EPSG:4326")  # WGS84

In [16]:
# Reading in ZCTA boundaries
zcta_gdf = gpd.read_file("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Raw Data/zcta_shapefiles/tl_2020_us_zcta520.shp").to_crs("EPSG:3857") 
# projecting ZCTA shapefile to a metric CRS (for distance calculation)
zcta_gdf = zcta_gdf.to_crs("EPSG:3857")  # Web Mercator: units in meters
restaurants_gdf = gdf.to_crs("EPSG:3857")

In [17]:
#doing a spatial join between restaurants df and zcta df with geometry
restaurants_with_zcta = gpd.sjoin(restaurants_gdf, zcta_gdf[['ZCTA5CE20', 'geometry']], how="left", predicate="within")
restaurants_with_zcta = restaurants_with_zcta.rename(columns={"ZCTA5CE20": "PRIMARY_ZCTA"})

In [5]:
#extracting 5-digit ZCTA from 'LOCATION' column
df['ZCTA5'] =df['LOCATION'].str.extract(r'ZCTA5 (\d{5})')

In [18]:
#left outer join with geocoded address data
restaurants_withsvi = restaurants_with_zcta.merge(df, left_on="PRIMARY_ZCTA", right_on="ZCTA5", how="left")

In [ ]:
#auditing 
#test = restaurants_withsvi[['ZCTA5', 'PRIMARY_ZCTA']]


In [20]:
#exporting
restaurants_withsvi.to_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_withsvi.csv")